#### Common Subexpression Elimination

In [ ]:
import import_ipynb
from P0 import compileString

Generate the RISC-V assembly code of the following P0 program:

In [ ]:
compileString("""
var x, y, z: integer
program p1
  z := x + 3
  x := (x + y) × (x + 3) + (x + y)
""", target = 'riscv')

In the generated code, which instructions reuse common subexpressions? State those instructions and state which subexpression is reused by each one!

CGriscv allocates scratch registers via `regs.pop()` on the `GPregs` set, so the exact `s`-register names vary per run; the structural pattern is deterministic. Write the generated instruction sequence abstractly as:

```
la   Rax, x_
lw   Rx,  0(Rax)           # Rx  := x                  (load x once)
addi R3,  Rx, 3            # R3  := x + 3
la   Raz, z_
sw   R3,  0(Raz)           # z   := x + 3              ← reuses x + 3
la   Ray, y_
lw   Ry,  0(Ray)           # Ry  := y
add  Rxy, Rx, Ry           # Rxy := x + y              ← reuses x (in Rx)
mul  Rm,  Rxy, R3          # Rm  := (x + y) × (x + 3)  ← reuses x + y AND x + 3
add  Rs,  Rm,  Rxy         # Rs  := Rm + (x + y)       ← reuses x + y
la   Rax2, x_
sw   Rs,  0(Rax2)          # x   := ...
```

Three common subexpressions are reused:

1. **`x`** (loaded once into `Rx` by `lw Rx, 0(Rax)`)
   - reused by `addi R3, Rx, 3` — first operand of `x + 3`
   - reused by `add Rxy, Rx, Ry` — first operand of `x + y`

2. **`x + 3`** (held in `R3`, produced once by `addi R3, Rx, 3`)
   - reused by `sw R3, 0(Raz)` — storing `z := x + 3`
   - reused by `mul Rm, Rxy, R3` — the `(x + 3)` factor of the product

3. **`x + y`** (held in `Rxy`, produced once by `add Rxy, Rx, Ry`)
   - reused by `mul Rm, Rxy, R3` — the `(x + y)` factor of the product
   - reused by `add Rs, Rm, Rxy` — the second occurrence of `(x + y)`

No instance of `x + 3` or `x + y` is recomputed; each is evaluated once and threaded through subsequent uses via its register. Whatever concrete `s2..s11` names `compileString(..., target='riscv')` prints on a given run, the three reuse patterns above hold.
